# Symbolic verification of the `curl_*_force_*` diagnostics

This notebook independently re-derives, with `sympy`, the true spherical curl
of each force in `Diagnostics_Curl_Momentum.F90` from that force's own
baseline formula (as coded in `Diagnostics_Linear_Forces.F90`,
`Diagnostics_Curl_Momentum.F90`'s `Grad_Viscous_Force`, etc.), and then checks
the *actual* Fortran expression coded for the corresponding `curl_*` quantity
against that derivation term-by-term.

For every force below, `report()` prints `MATCH` when
`sympy.simplify(true_curl - code_expression)` is identically zero, and prints
the (non-zero) residual otherwise.

Convention: `theta` is the polar angle from the rotation axis and `phi` is
the azimuthal angle, matching Rayleigh's spherical coordinate convention.
The Fortran-side names used in comparisons below (`buffer(PSI,...)`,
`DDBUFF(PSI,...)`, `one_over_r(r)`, `costheta(t)`, ...) are written as plain
`sympy` symbols with those exact string names, so the printed "code"
expressions can be read directly against the `.F90` source.

## Setup

Standard spherical curl:
$$(\nabla\times F)_r=\frac{1}{r\sin\theta}\left[\partial_\theta(F_\phi\sin\theta)-\partial_\phi F_\theta\right]$$
$$(\nabla\times F)_\theta=\frac{1}{r}\left[\frac{1}{\sin\theta}\partial_\phi F_r-\partial_r(rF_\phi)\right]$$
$$(\nabla\times F)_\phi=\frac{1}{r}\left[\partial_r(rF_\theta)-\partial_\theta F_r\right]$$

In [1]:

import re
import sympy as sp
sp.init_printing(use_latex='mathjax')

r, theta, phi = sp.symbols('r theta phi', positive=True)

def curl(Fr, Ft, Fp):
    # Standard spherical curl. theta=polar angle from axis, phi=azimuth.
    curl_r = (1/(r*sp.sin(theta))) * (sp.diff(Fp*sp.sin(theta), theta) - sp.diff(Ft, phi))
    curl_t = (1/r) * (sp.diff(Fr, phi)/sp.sin(theta) - sp.diff(r*Fp, r))
    curl_p = (1/r) * (sp.diff(r*Ft, r) - sp.diff(Fr, theta))
    return sp.simplify(curl_r), sp.simplify(curl_t), sp.simplify(curl_p)

def report(name, computed, coded):
    diff = sp.simplify(sp.expand_trig(sp.expand(computed - coded)))
    ok = diff == 0
    print(f"{name}: {'MATCH' if ok else 'MISMATCH'}")
    if not ok:
        print(f"   true - code = {diff}")
    return ok

# ---------------------------------------------------------------------
# Fortran-style pretty printer: same macro-aware line-splitting logic
# used to generate the actual qty(PSI) = ... assignments in
# Diagnostics_Curl_Momentum.F90 (DDBUFF -> d2buffer%p3a and PSI -> k,r,t
# expand under the C preprocessor, inflating the true line length beyond
# what the raw source shows -- so wrapping must budget for that expansion,
# not the raw string length).
# ---------------------------------------------------------------------
def _expand_macros(s):
    s2 = re.sub(r'\bDDBUFF\b', 'd2buffer%p3a', s)
    s2 = re.sub(r'\bPSI\b', 'k,r,t', s2)
    return s2

def _expanded_len(s):
    return len(_expand_macros(s))

def fortran_lines(expr, lhs='qty(PSI) = ', indent=' '*16, maxlen=126):
    terms = sp.Add.make_args(sp.expand(expr))
    pieces = []
    for t in terms:
        s = sp.sstr(t)
        if s.startswith('-'):
            pieces.append(('-', s[1:].strip()))
        else:
            pieces.append(('+', s))

    lines = []
    cur = None
    for idx, (sign, body) in enumerate(pieces):
        if idx == 0:
            cur = indent + lhs + (('-' if sign == '-' else '') + body)
        else:
            candidate = cur + f' {sign} ' + body
            if _expanded_len(candidate + ' &') <= maxlen:
                cur = candidate
            else:
                lines.append(cur + ' &')
                cur = indent + f'{sign} ' + body
    lines.append(cur)
    return '\n'.join(lines)


## 1. Coriolis force

Baseline (`Diagnostics_Linear_Forces.F90`, `Compute_Coriolis_Force`), with
$C=$`ref%Coriolis_Coeff` and $\rho=$`ref%density(r)`:

$$F_r = C\rho\sin\theta\,v_\phi,\qquad F_\theta = C\rho\cos\theta\,v_\phi,\qquad
F_\phi = -C\rho(\cos\theta\,v_\theta + \sin\theta\,v_r)$$

In [2]:

C = sp.Symbol('C')  # ref%Coriolis_Coeff
rho = sp.Function('rho')(r)
dlnrho = sp.diff(rho, r)/rho

vr = sp.Function('v_r')(r, theta, phi)
vt = sp.Function('v_t')(r, theta, phi)
vp = sp.Function('v_p')(r, theta, phi)

dvrdr, dvrdt, dvrdp = sp.diff(vr,r), sp.diff(vr,theta), sp.diff(vr,phi)
dvtdr, dvtdt, dvtdp = sp.diff(vt,r), sp.diff(vt,theta), sp.diff(vt,phi)
dvpdr, dvpdt, dvpdp = sp.diff(vp,r), sp.diff(vp,theta), sp.diff(vp,phi)

Fr_cor = C*rho*sp.sin(theta)*vp
Ft_cor = C*rho*sp.cos(theta)*vp
Fp_cor = -C*rho*(sp.cos(theta)*vt + sp.sin(theta)*vr)

Fr_cor, Ft_cor, Fp_cor


(C⋅ρ(r)⋅vₚ(r, θ, φ)⋅sin(θ), C⋅ρ(r)⋅vₚ(r, θ, φ)⋅cos(θ), -C⋅(vᵣ(r, θ, φ)⋅sin(θ) 
+ vₜ(r, θ, φ)⋅cos(θ))⋅ρ(r))

### True curl (symbolic)

In [3]:

cr_cor, ct_cor, cp_cor = curl(Fr_cor, Ft_cor, Fp_cor)
cr_cor, ct_cor, cp_cor


⎛  ⎛                                                                          
⎜  ⎜                                                                          
⎜  ⎜                                               vₜ(r, θ, φ)          ∂     
⎜C⋅⎜-2⋅vᵣ(r, θ, φ)⋅cos(θ) + 2⋅vₜ(r, θ, φ)⋅sin(θ) - ─────────── - sin(θ)⋅──(vᵣ(
⎜  ⎝                                                  sin(θ)            ∂θ    
⎜─────────────────────────────────────────────────────────────────────────────
⎝                                                                   r         

                                     ∂              ⎞                         
                                     ──(vₚ(r, θ, φ))⎟                         
                   ∂                 ∂φ             ⎟         ⎛               
r, θ, φ)) - cos(θ)⋅──(vₜ(r, θ, φ)) - ───────────────⎟⋅ρ(r)  C⋅⎜r⋅(vᵣ(r, θ, φ)⋅
                   ∂θ                     tan(θ)    ⎠         ⎝               
───────────────────────────────────────────────────

### Compare against the current Fortran (`Compute_Curl_Coriolis_Force`, lines 614-654)

In [4]:

oor = 1/r
csc = 1/sp.sin(theta)
cot = sp.cos(theta)/sp.sin(theta)
cos_ = sp.cos(theta)
sin_ = sp.sin(theta)

# curl_coriolis_force_r
code_r_cor = - C*rho*oor*(-sin_*vt + cot*cos_*vt + cos_*dvtdt + 2*cos_*vr + sin_*dvrdt + cot*dvpdp)
report("curl_coriolis_force_r", cr_cor, code_r_cor)

# curl_coriolis_force_theta
code_t_cor = C*rho*(oor*(dvpdp + cos_*vt + sin_*vr) + cos_*dvtdr + sin_*dvrdr + dlnrho*cos_*vt + dlnrho*sin_*vr)
report("curl_coriolis_force_theta", ct_cor, code_t_cor)

# curl_coriolis_force_phi
code_p_cor = C*rho*(dlnrho*cos_*vp + cos_*dvpdr - oor*sin_*dvpdt)
report("curl_coriolis_force_phi", cp_cor, code_p_cor)


curl_coriolis_force_r: MATCH
curl_coriolis_force_theta: MATCH
curl_coriolis_force_phi: MATCH


True

## 2. Buoyancy force

Baseline (`Diagnostics_Linear_Forces.F90`, `Compute_Buoyancy_Force`) is purely
radial, with $B_c(r)=$`ref%Buoyancy_Coeff(r)` and $T=$`buffer(PSI,tvar)`:

$$F_r = B_c(r)\,(T - T_0(r)),\qquad F_\theta = F_\phi = 0$$

so $(\nabla\times F)_r \equiv 0$ (matches the code, which has no
`curl_buoyancy_force_r`).

In [5]:

Bc = sp.Function('Bc')(r)
T  = sp.Function('T')(r, theta, phi)
T0 = sp.Function('T0')(r)
dtdt, dtdp = sp.diff(T, theta), sp.diff(T, phi)

Fr_b = Bc*(T - T0)
Ft_b = sp.Integer(0)
Fp_b = sp.Integer(0)

cr_b, ct_b, cp_b = curl(Fr_b, Ft_b, Fp_b)
print("curl_r (true):", cr_b, " -- always 0, as expected")
ct_b, cp_b


curl_r (true): 0  -- always 0, as expected


⎛      ∂                      ∂              ⎞
⎜Bc(r)⋅──(T(r, θ, φ))  -Bc(r)⋅──(T(r, θ, φ)) ⎟
⎜      ∂φ                     ∂θ             ⎟
⎜────────────────────, ──────────────────────⎟
⎝      r⋅sin(θ)                  r           ⎠

### Compare against the current Fortran (`Compute_Curl_Buoyancy_Force`, lines 278-305)

In [6]:

code_t_b = Bc * csc * oor * dtdp
report("curl_buoyancy_force_theta", ct_b, code_t_b)

code_p_b = -Bc * oor * dtdt
report("curl_buoyancy_force_phi", cp_b, code_p_b)


curl_buoyancy_force_theta: MATCH
curl_buoyancy_force_phi: MATCH


True

## 3. Pressure force

Baseline (`Diagnostics_Linear_Forces.F90`, `Compute_Pressure_Force`), with
`pfactor = ref%dpdr_w_term(r)/ref%density(r)` (the code's own comment notes
`pfactor` is treated as constant for this curl derivation) and
$\lambda(r)=$`ref%dlnrho(r)`:

$$F_r = -\text{pfactor}\,\partial_r(P-P_0) + \text{pfactor}\,\lambda(r)(P-P_0),\qquad
F_\theta = -\frac{\text{pfactor}}{r}\partial_\theta P,\qquad
F_\phi = -\frac{\text{pfactor}}{r\sin\theta}\partial_\phi P$$

In [7]:

pfactor = sp.Symbol('p_f')
dlnrho = sp.Function('dlnrho')(r)
P  = sp.Function('P')(r, theta, phi)
P0 = sp.Function('P0')(r)
dpdt, dpdp = sp.diff(P, theta), sp.diff(P, phi)

Fr_p = -pfactor*(sp.diff(P, r) - sp.diff(P0, r)) + pfactor*dlnrho*(P - P0)
Ft_p = -pfactor*dpdt/r
Fp_p = -pfactor*dpdp/(r*sp.sin(theta))

cr_p, ct_p, cp_p = curl(Fr_p, Ft_p, Fp_p)
print("curl_r (true):", cr_p, " -- always 0, as expected")
ct_p, cp_p


curl_r (true): 0  -- always 0, as expected


⎛              ∂                              ∂              ⎞
⎜p_f⋅dlnrho(r)⋅──(P(r, θ, φ))  -p_f⋅dlnrho(r)⋅──(P(r, θ, φ)) ⎟
⎜              ∂φ                             ∂θ             ⎟
⎜────────────────────────────, ──────────────────────────────⎟
⎝          r⋅sin(θ)                          r               ⎠

### Compare against the current Fortran (`Compute_Curl_Pressure_Force`, lines 693-721)

`curl_pressure_force_theta` needs the $\phi$-derivative of $P$ (`dpdp`), while
`curl_pressure_force_phi` needs the $\theta$-derivative (`dpdt`) — that
asymmetry is correct, not a typo (it falls straight out of the curl
formula).

In [8]:

code_t_p = pfactor * oor * csc * dlnrho * dpdp
report("curl_pressure_force_theta", ct_p, code_t_p)

code_p_p = -pfactor * oor * dlnrho * dpdt
report("curl_pressure_force_phi", cp_p, code_p_p)


curl_pressure_force_theta: MATCH
curl_pressure_force_phi: MATCH


True

## 4. Viscous force

`Grad_Viscous_Force` computes first derivatives of the viscous force itself
(stored via the `VFDBUFF`/`vforce_buffer` arrays as a generic vector field
$(v\!f_r, v\!f_\theta, v\!f_\phi)$) — so no extra baseline substitution is
needed here, `curl_viscous_force_*` is simply the generic spherical curl of
that vector field.

In [9]:

vfr = sp.Function('vfr')(r, theta, phi)
vft = sp.Function('vft')(r, theta, phi)
vfp = sp.Function('vfp')(r, theta, phi)

cr_v, ct_v, cp_v = curl(vfr, vft, vfp)
cr_v, ct_v, cp_v


⎛                                  ∂                                          
⎜                                  ──(vft(r, θ, φ))                           
⎜vfp(r, θ, φ)   ∂                  ∂φ                    ∂                    
⎜──────────── + ──(vfp(r, θ, φ)) - ────────────────  - r⋅──(vfp(r, θ, φ)) - vf
⎜   tan(θ)      ∂θ                      sin(θ)           ∂r                   
⎜──────────────────────────────────────────────────, ─────────────────────────
⎝                        r                                                    

             ∂                                                                
             ──(vfr(r, θ, φ))                                                 
             ∂φ                  ∂                                 ∂          
p(r, θ, φ) + ────────────────  r⋅──(vft(r, θ, φ)) + vft(r, θ, φ) - ──(vfr(r, θ
                  sin(θ)         ∂r                                ∂θ         
─────────────────────────────, ────────────────────

### Compare against the current Fortran (`Compute_Curl_Viscous_Force`, lines 732-775)

In [10]:

code_r_v = oor*(sp.diff(vfp, theta) + cot*vfp - csc*sp.diff(vft, phi))
report("curl_viscous_force_r", cr_v, code_r_v)

code_t_v = oor*(csc*sp.diff(vfr, phi) - vfp) - sp.diff(vfp, r)
report("curl_viscous_force_theta", ct_v, code_t_v)

code_p_v = sp.diff(vft, r) + oor*(vft - sp.diff(vfr, theta))
report("curl_viscous_force_phi", cp_v, code_p_v)


curl_viscous_force_r: MATCH
curl_viscous_force_theta: MATCH
curl_viscous_force_phi: MATCH


True

## 5. Advection force $\rho\,(v\cdot\nabla)v$ (`v_grad_v`)

Baseline (`Diagnostics_Inertial_Forces.F90`, and the underlying
`ADotGradB_3D3D` dispatch used at the `v_grad_v` call site), with
$\rho=$`ref%density(r)` and the standard spherical $(v\cdot\nabla)v$:

$$(v\cdot\nabla v)_r = v_r\partial_r v_r + \frac{v_\theta}{r}\partial_\theta v_r
   + \frac{v_\phi}{r\sin\theta}\partial_\phi v_r - \frac{v_\theta^2+v_\phi^2}{r}$$
$$(v\cdot\nabla v)_\theta = v_r\partial_r v_\theta + \frac{v_\theta}{r}\partial_\theta v_\theta
   + \frac{v_\phi}{r\sin\theta}\partial_\phi v_\theta + \frac{v_rv_\theta}{r} - \frac{v_\phi^2\cot\theta}{r}$$
$$(v\cdot\nabla v)_\phi = v_r\partial_r v_\phi + \frac{v_\theta}{r}\partial_\theta v_\phi
   + \frac{v_\phi}{r\sin\theta}\partial_\phi v_\phi + \frac{v_rv_\phi}{r} + \frac{v_\theta v_\phi\cot\theta}{r}$$

$F = \rho\,(v\cdot\nabla)v$ is what is curled below.

In [11]:

vgv_r = vr*dvrdr + vt*oor*dvrdt + vp*oor*csc*dvrdp - (vt**2+vp**2)*oor
vgv_t = vr*dvtdr + vt*oor*dvtdt + vp*oor*csc*dvtdp + vr*vt*oor - vp**2*cot*oor
vgv_p = vr*dvpdr + vt*oor*dvpdt + vp*oor*csc*dvpdp + vr*vp*oor + vt*vp*cot*oor

Fr_a = rho*vgv_r
Ft_a = rho*vgv_t
Fp_a = rho*vgv_p

cr_a, ct_a, cp_a = curl(Fr_a, Ft_a, Fp_a)
cr_a, ct_a, cp_a


⎛                                                                             
⎜                                                                             
⎜                                                                             
⎜⎛  ⎛               2                                                ⎞        
⎜⎜  ⎜              ∂                  ∂               ∂              ⎟    2   
⎜⎜r⋅⎜vᵣ(r, θ, φ)⋅─────(vₚ(r, θ, φ)) + ──(vₚ(r, θ, φ))⋅──(vᵣ(r, θ, φ))⎟⋅sin (θ)
⎜⎜  ⎝            ∂θ ∂r                ∂r              ∂θ             ⎠        
⎜⎝                                                                            
⎜─────────────────────────────────────────────────────────────────────────────
⎜                                                                             
⎝                                                                             

                                                                              
                                                   

### Fortranize: substitute sympy `Derivative`/`Function` objects with symbols
named exactly like the Fortran buffer accessors, so the printed result can be
read straight off `Diagnostics_Curl_Momentum.F90`.

The substitution list is applied to $(\nabla\times F)_r,\ (\nabla\times F)_\theta,\
(\nabla\times F)_\phi$ above, and a completeness check confirms no
`Derivative`/`Function` objects are left over (i.e. the substitution was
lossless) before trusting the printed Fortran-symbol form.

In [12]:

fields = {'vr': vr, 'vt': vt, 'vp': vp}
field_buffer_name = {'vr': 'vr', 'vt': 'vtheta', 'vp': 'vphi'}   # buffer(PSI,<name>)
varsym = {'r': r, 't': theta, 'p': phi}

subs_list = []

# second derivatives first (must precede first-derivative substitution, since
# first-derivative patterns are sub-expressions of second derivatives)
pairs = [('r','r'), ('r','t'), ('r','p'), ('t','t'), ('t','p'), ('p','p')]
for fshort, ffunc in fields.items():
    for a, b in pairs:
        name = f'DDBUFF(PSI,d{fshort}d{a}d{b})'
        subs_list.append((sp.diff(ffunc, varsym[a], varsym[b]), sp.Symbol(name)))

# first derivatives
for fshort, ffunc in fields.items():
    for a in ['r','t','p']:
        name = f'buffer(PSI,d{fshort}d{a})'
        subs_list.append((sp.diff(ffunc, varsym[a]), sp.Symbol(name)))

# bare field values
for fshort, ffunc in fields.items():
    name = f'buffer(PSI,{field_buffer_name[fshort]})'
    subs_list.append((ffunc, sp.Symbol(name)))

# density: substitute the derivative FIRST (rho' = dlnrho*rho), then rho itself
RHO = sp.Symbol('ref%density(r)')
DLNRHO = sp.Symbol('ref%dlnrho(r)')
subs_list = [(sp.diff(rho, r), DLNRHO*RHO)] + subs_list + [(rho, RHO)]

# metric factors
OOR = sp.Symbol('one_over_r(r)')
CSC = sp.Symbol('csctheta(t)')
COT = sp.Symbol('cottheta(t)')
COS_ = sp.Symbol('costheta(t)')
metric_subs = [(oor, OOR), (csc, CSC), (cot, COT), (cos_, COS_),
               (1/sp.tan(theta), COT), (sp.tan(theta), 1/COT)]

def fortranize(expr):
    e = expr.subs(subs_list)
    e = sp.expand(e)
    e = e.subs(metric_subs)
    return e

fr_out = fortranize(cr_a)
ft_out = fortranize(ct_a)
fp_out = fortranize(cp_a)

for name, e in [("curl_v_grad_v_r", fr_out), ("curl_v_grad_v_theta", ft_out), ("curl_v_grad_v_phi", fp_out)]:
    residual_atoms = [a for a in e.atoms(sp.Function, sp.Derivative) if not isinstance(a, sp.Symbol)]
    print(f"{name}: {'CLEAN (fully expressed in Fortran symbols)' if not residual_atoms else 'LEFTOVER: '+str(residual_atoms)}")


curl_v_grad_v_r: CLEAN (fully expressed in Fortran symbols)
curl_v_grad_v_theta: CLEAN (fully expressed in Fortran symbols)
curl_v_grad_v_phi: CLEAN (fully expressed in Fortran symbols)


### `curl_v_grad_v_r` (Fortran-symbol form)

This is exactly the expression now coded at
`Diagnostics_Curl_Momentum.F90`'s `curl_v_grad_v_r` block (17 additive terms,
matching one-for-one with the lines of that `qty(PSI) = ...` assignment).

In [13]:
print(fortran_lines(fr_out))

                qty(PSI) = DDBUFF(PSI,dvpdrdt)*buffer(PSI,vr)*one_over_r(r)*ref%density(r) &
                + DDBUFF(PSI,dvpdtdt)*buffer(PSI,vtheta)*one_over_r(r)**2*ref%density(r) &
                + buffer(PSI,dvpdr)*buffer(PSI,dvrdt)*one_over_r(r)*ref%density(r) &
                + buffer(PSI,dvpdt)*buffer(PSI,dvtdt)*one_over_r(r)**2*ref%density(r) &
                + buffer(PSI,dvpdt)*buffer(PSI,vr)*one_over_r(r)**2*ref%density(r) &
                + buffer(PSI,dvrdt)*buffer(PSI,vphi)*one_over_r(r)**2*ref%density(r) &
                - buffer(PSI,vphi)*buffer(PSI,vtheta)*one_over_r(r)**2*ref%density(r) &
                + DDBUFF(PSI,dvpdtdp)*buffer(PSI,vphi)*csctheta(t)*one_over_r(r)**2*ref%density(r) &
                + buffer(PSI,dvpdp)*buffer(PSI,dvpdt)*csctheta(t)*one_over_r(r)**2*ref%density(r) &
                - DDBUFF(PSI,dvtdpdp)*buffer(PSI,vphi)*csctheta(t)**2*one_over_r(r)**2*ref%density(r) &
                - DDBUFF(PSI,dvtdrdp)*buffer(PSI,vr)*csctheta(t)*one_over_r(r)

### `curl_v_grad_v_theta` (Fortran-symbol form)

In [14]:
print(fortran_lines(ft_out))

                qty(PSI) = -DDBUFF(PSI,dvpdrdr)*buffer(PSI,vr)*ref%density(r) &
                - buffer(PSI,dvpdr)*buffer(PSI,dvrdr)*ref%density(r) &
                - DDBUFF(PSI,dvpdrdt)*buffer(PSI,vtheta)*one_over_r(r)*ref%density(r) &
                - buffer(PSI,dvpdr)*buffer(PSI,vr)*ref%density(r)*ref%dlnrho(r) &
                - buffer(PSI,dvpdt)*buffer(PSI,dvtdr)*one_over_r(r)*ref%density(r) &
                - buffer(PSI,dvrdr)*buffer(PSI,vphi)*one_over_r(r)*ref%density(r) &
                - 2*buffer(PSI,dvpdr)*buffer(PSI,vr)*one_over_r(r)*ref%density(r) &
                + DDBUFF(PSI,dvrdpdp)*buffer(PSI,vphi)*csctheta(t)**2*one_over_r(r)**2*ref%density(r) &
                + DDBUFF(PSI,dvrdrdp)*buffer(PSI,vr)*csctheta(t)*one_over_r(r)*ref%density(r) &
                + DDBUFF(PSI,dvrdtdp)*buffer(PSI,vtheta)*csctheta(t)*one_over_r(r)**2*ref%density(r) &
                + buffer(PSI,dvpdp)*buffer(PSI,dvrdp)*csctheta(t)**2*one_over_r(r)**2*ref%density(r) &
                + bu

### `curl_v_grad_v_phi` (Fortran-symbol form)

In [15]:
print(fortran_lines(fp_out))

                qty(PSI) = DDBUFF(PSI,dvtdrdr)*buffer(PSI,vr)*ref%density(r) &
                + buffer(PSI,dvrdr)*buffer(PSI,dvtdr)*ref%density(r) &
                + DDBUFF(PSI,dvtdrdt)*buffer(PSI,vtheta)*one_over_r(r)*ref%density(r) &
                + buffer(PSI,dvrdr)*buffer(PSI,vtheta)*one_over_r(r)*ref%density(r) &
                + buffer(PSI,dvtdr)*buffer(PSI,dvtdt)*one_over_r(r)*ref%density(r) &
                + buffer(PSI,dvtdr)*buffer(PSI,vr)*ref%density(r)*ref%dlnrho(r) &
                - DDBUFF(PSI,dvrdrdt)*buffer(PSI,vr)*one_over_r(r)*ref%density(r) &
                - DDBUFF(PSI,dvrdtdt)*buffer(PSI,vtheta)*one_over_r(r)**2*ref%density(r) &
                - buffer(PSI,dvrdr)*buffer(PSI,dvrdt)*one_over_r(r)*ref%density(r) &
                - buffer(PSI,dvrdt)*buffer(PSI,dvtdt)*one_over_r(r)**2*ref%density(r) &
                + 2*buffer(PSI,dvpdt)*buffer(PSI,vphi)*one_over_r(r)**2*ref%density(r) &
                + 2*buffer(PSI,dvtdr)*buffer(PSI,vr)*one_over_r(r)*ref%d

### `curl_v_grad_v_abs`

Coded as $\rho(r)\sqrt{\hat r^2+\hat\theta^2+\hat\phi^2}$ where $\hat r,\hat\theta,\hat\phi$
are the `curl_v_grad_v_r/theta/phi` formulas above with the common factor
`ref%density(r)` divided out. Verify that factoring is exact (remainder is
identically zero for each component) before trusting the `vgv_abs_r/t/p`
local temporaries in the code.

In [16]:

for name, full in [("r", fr_out), ("theta", ft_out), ("phi", fp_out)]:
    bracket = sp.simplify(sp.expand(full)/RHO)
    residual = sp.simplify(full - RHO*bracket)
    print(f"{name}: density factors out cleanly = {residual == 0}")


r: density factors out cleanly = True
theta: density factors out cleanly = True
phi: density factors out cleanly = True


## 6. Magnetic (Lorentz) force $L_c\,(\nabla\times B)\times B$ (`j_cross_b`)

Baseline (`Diagnostics_Lorentz_Forces.F90`), with $L_c=$`ref%Lorentz_Coeff`
and $J=\nabla\times B$ (the current density, up to the constant folded into
$L_c$):

$$F = L_c\,(\nabla\times B)\times B,\qquad
F_r = L_c(J_\theta B_\phi - J_\phi B_\theta),\quad
F_\theta = L_c(J_\phi B_r - J_r B_\phi),\quad
F_\phi = L_c(J_r B_\theta - J_\theta B_r)$$

In [17]:

Lc = sp.Symbol('L_c')
Br = sp.Function('B_r')(r, theta, phi)
Bt = sp.Function('B_t')(r, theta, phi)
Bp = sp.Function('B_p')(r, theta, phi)

dbrdr, dbrdt, dbrdp = sp.diff(Br,r), sp.diff(Br,theta), sp.diff(Br,phi)
dbtdr, dbtdt, dbtdp = sp.diff(Bt,r), sp.diff(Bt,theta), sp.diff(Bt,phi)
dbpdr, dbpdt, dbpdp = sp.diff(Bp,r), sp.diff(Bp,theta), sp.diff(Bp,phi)

Jr, Jt, Jp = curl(Br, Bt, Bp)

Fr_j = Lc*(Jt*Bp - Jp*Bt)
Ft_j = Lc*(Jp*Br - Jr*Bp)
Fp_j = Lc*(Jr*Bt - Jt*Br)

Fr_j, Ft_j, Fp_j


⎛    ⎛⎛                                    ∂              ⎞                   
⎜    ⎜⎜                                    ──(Bᵣ(r, θ, φ))⎟                   
⎜    ⎜⎜    ∂                               ∂φ             ⎟               ⎛  ∂
⎜    ⎜⎜- r⋅──(Bₚ(r, θ, φ)) - Bₚ(r, θ, φ) + ───────────────⎟⋅Bₚ(r, θ, φ)   ⎜r⋅─
⎜    ⎜⎝    ∂r                                   sin(θ)    ⎠               ⎝  ∂
⎜L_c⋅⎜───────────────────────────────────────────────────────────────── - ────
⎝    ⎝                                r                                       

                                                           ⎞      ⎛           
                                                           ⎟      ⎜           
                               ∂              ⎞            ⎟      ⎜⎛  ∂       
─(Bₜ(r, θ, φ)) + Bₜ(r, θ, φ) - ──(Bᵣ(r, θ, φ))⎟⋅Bₜ(r, θ, φ)⎟      ⎜⎜r⋅──(Bₜ(r,
r                              ∂θ             ⎠            ⎟      ⎜⎝  ∂r      
───────────────────────────────────────────────────

### True curl (symbolic)

$\nabla\times[L_c(\nabla\times B)\times B]$ involves up to second derivatives
of $B$ (one derivative from the inner curl, one more from the outer curl),
matching the `DDBUFF` second-derivative terms already used in the code for
this diagnostic.

In [18]:

cr_j, ct_j, cp_j = curl(Fr_j, Ft_j, Fp_j)
cr_j, ct_j, cp_j


⎛    ⎛                                                                        
⎜    ⎜                                                                        
⎜    ⎜                                                 ∂                      
⎜    ⎜                 2                 r⋅Bᵣ(r, θ, φ)⋅──(Bₚ(r, θ, φ))   r⋅Bᵣ(
⎜    ⎜                ∂                                ∂r                     
⎜L_c⋅⎜r⋅Bᵣ(r, θ, φ)⋅─────(Bₚ(r, θ, φ)) + ───────────────────────────── - ─────
⎜    ⎜              ∂θ ∂r                            tan(θ)                   
⎜    ⎝                                                                        
⎜─────────────────────────────────────────────────────────────────────────────
⎜                                                                             
⎝                                                                             

                                                                              
            2                                      

### Fortranize

Same procedure as for `v_grad_v`: substitute `Derivative`/`Function` objects
for symbols named exactly like the Fortran buffer accessors, then confirm
the substitution was lossless.

In [19]:

fields_j = {'br': Br, 'bt': Bt, 'bp': Bp}
field_buffer_name_j = {'br': 'br', 'bt': 'btheta', 'bp': 'bphi'}

subs_list_j = []
for fshort, ffunc in fields_j.items():
    for a, b in pairs:
        name = f'DDBUFF(PSI,d{fshort}d{a}d{b})'
        subs_list_j.append((sp.diff(ffunc, varsym[a], varsym[b]), sp.Symbol(name)))
for fshort, ffunc in fields_j.items():
    for a in ['r','t','p']:
        name = f'buffer(PSI,d{fshort}d{a})'
        subs_list_j.append((sp.diff(ffunc, varsym[a]), sp.Symbol(name)))
for fshort, ffunc in fields_j.items():
    name = f'buffer(PSI,{field_buffer_name_j[fshort]})'
    subs_list_j.append((ffunc, sp.Symbol(name)))

LC = sp.Symbol('ref%Lorentz_Coeff')
metric_subs_j = [(oor, OOR), (csc, CSC), (cot, COT), (cos_, COS_),
                  (1/sp.tan(theta), COT), (sp.tan(theta), 1/COT), (Lc, LC)]

def fortranize_j(expr):
    e = expr.subs(subs_list_j)
    e = sp.expand(e)
    e = e.subs(metric_subs_j)
    e = sp.expand(e)
    return e

fr_out_j = fortranize_j(cr_j)
ft_out_j = fortranize_j(ct_j)
fp_out_j = fortranize_j(cp_j)

for name, e in [("curl_j_cross_b_r", fr_out_j), ("curl_j_cross_b_theta", ft_out_j), ("curl_j_cross_b_phi", fp_out_j)]:
    residual_atoms = [a for a in e.atoms(sp.Function, sp.Derivative) if not isinstance(a, sp.Symbol)]
    print(f"{name}: {'CLEAN (fully expressed in Fortran symbols)' if not residual_atoms else 'LEFTOVER: '+str(residual_atoms)}")


curl_j_cross_b_r: CLEAN (fully expressed in Fortran symbols)
curl_j_cross_b_theta: CLEAN (fully expressed in Fortran symbols)
curl_j_cross_b_phi: CLEAN (fully expressed in Fortran symbols)


### `curl_j_cross_b_r` (Fortran-symbol form)

This is exactly the expression now coded at
`Diagnostics_Curl_Momentum.F90`'s `curl_j_cross_b_r` block.

In [20]:
print(fortran_lines(fr_out_j))

                qty(PSI) = DDBUFF(PSI,dbpdrdt)*buffer(PSI,br)*one_over_r(r)*ref%Lorentz_Coeff &
                + DDBUFF(PSI,dbpdtdt)*buffer(PSI,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,bphi)*buffer(PSI,dbrdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,br)*buffer(PSI,dbpdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,dbpdr)*buffer(PSI,dbrdt)*one_over_r(r)*ref%Lorentz_Coeff &
                + buffer(PSI,dbpdt)*buffer(PSI,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                - buffer(PSI,bphi)*buffer(PSI,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + DDBUFF(PSI,dbpdtdp)*buffer(PSI,bphi)*csctheta(t)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,bphi)*buffer(PSI,br)*cottheta(t)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,bphi)*buffer(PSI,dbtdt)*cottheta(t)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,br)*buffer(PSI,dbpdr)*cotthe

### `curl_j_cross_b_theta` (Fortran-symbol form)

In [21]:
print(fortran_lines(ft_out_j))

                qty(PSI) = -DDBUFF(PSI,dbpdrdr)*buffer(PSI,br)*ref%Lorentz_Coeff &
                - buffer(PSI,dbpdr)*buffer(PSI,dbrdr)*ref%Lorentz_Coeff &
                - DDBUFF(PSI,dbpdrdt)*buffer(PSI,btheta)*one_over_r(r)*ref%Lorentz_Coeff &
                - buffer(PSI,bphi)*buffer(PSI,dbrdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - buffer(PSI,dbpdt)*buffer(PSI,dbtdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - 2*buffer(PSI,br)*buffer(PSI,dbpdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + DDBUFF(PSI,dbrdpdp)*buffer(PSI,bphi)*csctheta(t)**2*one_over_r(r)**2*ref%Lorentz_Coeff &
                + DDBUFF(PSI,dbrdrdp)*buffer(PSI,br)*csctheta(t)*one_over_r(r)*ref%Lorentz_Coeff &
                + DDBUFF(PSI,dbrdtdp)*buffer(PSI,btheta)*csctheta(t)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,dbpdp)*buffer(PSI,dbrdp)*csctheta(t)**2*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,dbrdp)*buffer(PSI,dbrdr)*csctheta(t)*one_ov

### `curl_j_cross_b_phi` (Fortran-symbol form)

In [22]:
print(fortran_lines(fp_out_j))

                qty(PSI) = DDBUFF(PSI,dbtdrdr)*buffer(PSI,br)*ref%Lorentz_Coeff &
                + buffer(PSI,dbrdr)*buffer(PSI,dbtdr)*ref%Lorentz_Coeff &
                + DDBUFF(PSI,dbtdrdt)*buffer(PSI,btheta)*one_over_r(r)*ref%Lorentz_Coeff &
                + buffer(PSI,btheta)*buffer(PSI,dbrdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + buffer(PSI,dbtdr)*buffer(PSI,dbtdt)*one_over_r(r)*ref%Lorentz_Coeff &
                - DDBUFF(PSI,dbrdrdt)*buffer(PSI,br)*one_over_r(r)*ref%Lorentz_Coeff &
                - DDBUFF(PSI,dbrdtdt)*buffer(PSI,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                - buffer(PSI,dbrdr)*buffer(PSI,dbrdt)*one_over_r(r)*ref%Lorentz_Coeff &
                - buffer(PSI,dbrdt)*buffer(PSI,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + 2*buffer(PSI,bphi)*buffer(PSI,dbpdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + 2*buffer(PSI,br)*buffer(PSI,dbtdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + 2*buffer(PSI,btheta

### `curl_j_cross_b_abs`

Coded as $L_c\sqrt{\hat r^2+\hat\theta^2+\hat\phi^2}$ where $\hat r,\hat\theta,\hat\phi$
are the `curl_j_cross_b_r/theta/phi` formulas above with the common factor
`ref%Lorentz_Coeff` divided out. Verify that factoring is exact before
trusting the `jxb_abs_r/t/p` local temporaries in the code.

In [23]:

for name, full in [("r", fr_out_j), ("theta", ft_out_j), ("phi", fp_out_j)]:
    bracket = sp.simplify(sp.expand(full)/LC)
    residual = sp.simplify(full - LC*bracket)
    print(f"{name}: Lorentz coefficient factors out cleanly = {residual == 0}")


r: Lorentz coefficient factors out cleanly = True
theta: Lorentz coefficient factors out cleanly = True
phi: Lorentz coefficient factors out cleanly = True
